### Import Dependencies

In [1]:
import uuid

import httpx
from a2a.client import A2ACardResolver, ClientFactory
from a2a.types import (
    AgentCard,
    Message,
    MessageSendParams,
    Part,
    Role,
    SendMessageRequest,
    TextPart,
)

In [2]:
BASE_URL = "http://localhost:10001"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent.json"

In [3]:
async with httpx.AsyncClient() as httpx_client:
    # Initialize A2ACardResolver
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL,
    )

    try:
        print(
            f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}"
        )
        _public_card = await resolver.get_agent_card()
        print("Fetched public agent card")
        print(_public_card.model_dump_json(indent=2))

        final_agent_card_to_use = _public_card

    except Exception as e:
        print(f"Error fetching public agent card: {e}")
        raise RuntimeError("Failed to fetch public agent card")

Fetching public agent card from: http://localhost:10001/.well-known/agent.json
Fetched public agent card
{
  "additionalInterfaces": null,
  "capabilities": {
    "extensions": null,
    "pushNotifications": null,
    "stateTransitionHistory": null,
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "A agent that can check the availability of items in the warehouses and reserve them.",
  "documentationUrl": null,
  "iconUrl": null,
  "name": "warehouse_manager_agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "provider": null,
  "security": null,
  "securitySchemes": null,
  "signatures": null,
  "skills": [
    {
      "description": "Check the availability of items in the warehouses",
      "examples": [
        "what is the availability of the item 123?"
      ],
      "id": "ABC",
      "inputModes": null,
      "name": "Check Availability",
      "outputModes": null,
      "securit

In [4]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await ClientFactory.connect(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.user,
    messageId=message_id,
    parts=[Part(root=TextPart(text="Hello, how are you?"))],
)

async for response in client.send_message(message_payload):
    print(response)


A2AClient initialized
(Task(artifacts=[Artifact(artifact_id='84002a3f-5226-4548-9070-0b0ba65106fe', description=None, extensions=None, metadata=None, name=None, parts=[Part(root=TextPart(kind='text', metadata=None, text="Hello! I'm here and ready to assist you with managing inventory and reservations in the warehouses. How can I help you today?"))])], context_id='681318ac-a840-4cb7-8f89-da9f8bbe4da9', history=None, id='2eab315d-4c90-42f6-b1e0-63298235e544', kind='task', metadata=None, status=TaskStatus(message=None, state=<TaskState.unknown: 'unknown'>, timestamp=None)), TaskArtifactUpdateEvent(append=None, artifact=Artifact(artifact_id='84002a3f-5226-4548-9070-0b0ba65106fe', description=None, extensions=None, metadata=None, name=None, parts=[Part(root=TextPart(kind='text', metadata=None, text="Hello! I'm here and ready to assist you with managing inventory and reservations in the warehouses. How can I help you today?"))]), context_id='681318ac-a840-4cb7-8f89-da9f8bbe4da9', kind='artif

In [5]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await ClientFactory.connect(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.user,
    messageId=message_id,
    parts=[Part(root=TextPart(text="Hello, how are you?"))],
)

print("Sending message")
final_response = None
async for response in client.send_message(message_payload):
    final_response = response

print("Response:")
for artifact in final_response[0].artifacts:
    for part in artifact.parts:
        print(part.root.text)

A2AClient initialized
Sending message
Response:
Hello! I'm here and ready to assist you with managing inventory and reservations in the warehouses. How can I help you today?


In [7]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await ClientFactory.connect(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.user,
    messageId=message_id,
    parts=[Part(root=TextPart(text="What is the availability of B09YCSP9FH in all of the warehouses?"))],
)

print("Sending message")
final_response = None
async for response in client.send_message(message_payload):
    final_response = response

print("Response:")
for artifact in final_response[0].artifacts:
    for part in artifact.parts:
        print(part.root.text)

A2AClient initialized
Sending message
Response:
The product B09YCSP9FH is available in the following warehouses with full fulfillment capability:

- Berlin Distribution Center (Berlin, Germany) with 29 units available
- Munich Logistics Hub (Munich, Germany) with 17 units available
- Paris Central Depot (Paris, France) with 88 units available
- Marseille Mediterranean Hub (Marseille, France) with 1 unit available

It is not available in Lyon Regional Warehouse (Lyon, France) and Hamburg North Warehouse (Hamburg, Germany). Let me know if you want to reserve any quantity from these warehouses.
